In [ ]:
#The purpose of this script is to relabel overstriding if I messed it up while labelling before
import cv2
import json
from collections import defaultdict
import sys, os
sys.path.insert(0, "/Users/abhinavarora/Desktop/CadenceCV/ml/src")
from utils import yolo_model_loader
import torch

with open("/Users/abhinavarora/Desktop/CadenceCV/ml/data/strikefoot_data.json", "r") as file:
    data = json.load(file)

#Dictionary for mapping frames to each video:
video_to_frames = defaultdict()

for frame_data in data:
    video_key = frame_data["video"]
    if video_key not in video_to_frames:
        video_to_frames[video_key] = []
    if video_key in video_to_frames:
        video_to_frames[video_key].append(frame_data["frame"])

print(video_to_frames)

base_video_dir_path = "/Users/abhinavarora/Desktop/CadenceCV/Videos"

video_extensions = [".mp4", ".MOV"]

relabelled_data = []

video_key_set = set(video_to_frames.keys())

In [ ]:
model = yolo_model_loader.load_model("/Users/abhinavarora/Desktop/CadenceCV/ml/weights/21_keypoints.pt")
video = next(iter(video_key_set))
video_key_set.remove(video)

if video is not None:
    try:
        for ext in video_extensions:
            cap = cv2.VideoCapture(base_video_dir_path + "/" + video + ext)
            frame_num = cap.get(cv2.CAP_PROP_FRAME_COUNT)
            frames = video_to_frames[video]

            for i in frames:
                ret, frame = cap.read()
                source = model.predict(frame, conf=0.5)
                for result in source:
                    kpts = result.keypoints.xy[0]

                for point in kpts:
                    p:torch.tensor = point
                    frame = cv2.circle(frame, tuple(p.int().tolist()), 2, (0,0,255), -1)

                cv2.imshow("Frame", frame)

                key = cv2.waitKey(0)
                #Cannot accept any other input
                while key != ord('o') and key != ord('n'):
                    key = cv2.waitKey(0)
                d = {"video": video, "frame": i, "label": chr(key)}
                relabelled_data.append(d)

            
            break

    except Exception as e:
        print(e)

cap.release()
cv2.destroyAllWindows()

In [ ]:
try:
    with open("/Users/abhinavarora/Desktop/CadenceCV/ml/data/overstriding_label_data.json", "r") as file:
        data = json.load(file)
except:
    data = []

for label in relabelled_data:
    data.append(label)

print(data)
with open("/Users/abhinavarora/Desktop/CadenceCV/ml/data/overstriding_label_data.json", "w") as file:
    json.dump(data, file, indent=4)

In [ ]:
with open("/Users/abhinavarora/Desktop/CadenceCV/ml/data/strikefoot_data.json", "r") as file:
    data = json.load(file)

strike_dict = {"heel": 0, "midfoot": 0, "forefoot": 0}
for d in data:
    strike_dict[d["strike_pattern"]] += 1

l = 0
for key in strike_dict:
    l += strike_dict[key]

for key in strike_dict:
    print(key + " ratio" + " " + str(strike_dict[key]/l))